# Multimodal 3D Generation - Google Colab Training

Notebook này giúp:
1. Mount Google Drive và thiết lập đường dẫn
2. Kiểm tra project code
3. Tải dữ liệu Cap3D (~40GB)
4. Train model với GPU T4
5. Auto-save checkpoint vào Drive
6. Evaluate và chạy demo

**Lưu ý**: Chọn Runtime → Change runtime type → **T4 GPU**

## 1. Setup & Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
    print('✓ Google Drive mounted')
except:
    print('✓ Google Drive already mounted')

# Kiểm tra GPU
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
    print('✓ GPU available')
except:
    print('⚠ KHÔNG CÓ GPU! Vui lòng chọn Runtime → Change runtime type → T4 GPU')

In [ ]:
import os

# === CẤU HÌNH ĐƯỜNG DẪN ===
# Mọi thứ chạy trực tiếp trên Drive
WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'

# Tạo thư mục nếu chưa có
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

print(f'✓ Work dir: {WORK_DIR}')

## 2. Kiểm tra Project Code

Project code nằm trực tiếp trên Drive.  
Đảm bảo đã upload các thư mục sau lên Drive: `src/`, `configs/`, `images/`

In [ ]:
# Kiểm tra xem có đủ files không
import os

required = ['src', 'configs', 'src/train.py', 'src/data/dataset.py', 'src/models/multimodal3d.py']
all_ok = True
for r in required:
    path = os.path.join(WORK_DIR, r)
    exists = os.path.exists(path)
    status = '✓' if exists else '✗ THIẾU'
    print(f'  {status}: {r}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n✓ Project structure OK!')
else:
    print('\n✗ Thiếu files! Upload src/ và configs/ lên Drive.')

## 3. Cài Dependencies

In [ ]:
# Cài các thư viện cần thiết
!pip install -q open-clip-torch trimesh gradio plotly tensorboard scipy tqdm pyyaml

## 4. Tải dữ liệu Cap3D

Dữ liệu sẽ được tải và giải nén trực tiếp lên Drive (~40GB).  
Nếu đã có data, cell này sẽ skip.

In [ ]:
%%time
import os, zipfile, shutil, io, json

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
os.makedirs(f'{DATA_DIR}/pointclouds', exist_ok=True)
os.makedirs(f'{DATA_DIR}/rendered_images', exist_ok=True)

# --- [1/3] Captions ---
csv_path = f'{DATA_DIR}/Cap3D_automated_ABO.csv'
if os.path.exists(csv_path):
    print('[1/3] ✓ Captions đã có')
else:
    print('[1/3] Tải captions...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D Cap3D_automated_ABO.csv \
        --repo-type dataset --local-dir /tmp/cap3d_download
    shutil.copy2('/tmp/cap3d_download/Cap3D_automated_ABO.csv', csv_path)
    print('✓ Captions saved')

# --- [2/3] Point clouds ---
pc_dir = f'{DATA_DIR}/pointclouds'
n_pc = len([f for f in os.listdir(pc_dir) if f.endswith('.ply')])
if n_pc > 100:
    print(f'\n[2/3] ✓ Point clouds đã có ({n_pc} files)')
else:
    print('\n[2/3] Tải point clouds (~1.5GB)...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D PointCloud_zips_ABO/compressed_pcs_00.zip \
        --repo-type dataset --local-dir /tmp/cap3d_download
    pc_zip = '/tmp/cap3d_download/PointCloud_zips_ABO/compressed_pcs_00.zip'
    print('Giải nén lên Drive...')
    with zipfile.ZipFile(pc_zip, 'r') as z:
        members = [m for m in z.namelist() if m.endswith('.ply')]
        for i, m in enumerate(members, 1):
            fname = os.path.basename(m)
            with z.open(m) as src, open(f'{pc_dir}/{fname}', 'wb') as dst:
                dst.write(src.read())
            if i % 2000 == 0 or i == len(members):
                print(f'  [{i}/{len(members)}]')
    os.remove(pc_zip)
    print(f'✓ Point clouds: {len(os.listdir(pc_dir))} files')

# --- [3/3] Rendered images (SMART RESUME) ---
img_dir = f'{DATA_DIR}/rendered_images'
existing_folders = set([d for d in os.listdir(img_dir) if os.path.isdir(f'{img_dir}/{d}')])

# Lấy danh sách object IDs từ captions
import pandas as pd
df = pd.read_csv(csv_path)
print(f'CSV columns: {list(df.columns)}')
# Tìm cột chứa object ID (có thể là 'object_id', 'object_id ', 'id', v.v.)
id_col = None
for col in df.columns:
    if 'object' in col.lower() or 'id' in col.lower():
        id_col = col
        break
if id_col is None:
    id_col = df.columns[0]  # Dùng cột đầu tiên nếu không tìm thấy
print(f'Using column: {id_col}')
expected_ids = set(df[id_col].astype(str).tolist())

missing_ids = expected_ids - existing_folders
n_existing = len(existing_folders)
n_missing = len(missing_ids)
n_total = len(expected_ids)

if n_missing == 0:
    print(f'\n[3/3] ✓ Rendered images đã có ({n_existing}/{n_total} folders)')
else:
    print(f'\n[3/3] Rendered images thiếu ({n_missing}/{n_total} folders). Tiếp tục tải...')
    !pip install -q huggingface_hub
    
    # Download main zip nếu chưa có trong /tmp
    main_zip_path = '/tmp/cap3d_download/RenderedImage_perobj_zips_ABO/compressed_imgs_perobj_00.zip'
    if not os.path.exists(main_zip_path):
        print('Đang tải main zip (~37GB)...')
        !hf download tiange/Cap3D RenderedImage_perobj_zips_ABO/compressed_imgs_perobj_00.zip \
            --repo-type dataset --local-dir /tmp/cap3d_download
        print('✓ Main zip downloaded')
    
    print(f'Giải nén {n_missing} folders còn thiếu...')
    with zipfile.ZipFile(main_zip_path, 'r') as outer_z:
        inner_zips = [m for m in outer_z.namelist() if m.endswith('.zip')]
        
        # Tạo map từ tên inner zip đến object_id
        zip_to_id = {z: os.path.basename(z).replace('.zip', '') for z in inner_zips}
        
        # Chỉ extract những zip cần thiết
        needed_zips = [z for z in inner_zips if zip_to_id[z] in missing_ids]
        print(f'  Cần giải nén {len(needed_zips)} inner zips...')
        
        for i, inner_name in enumerate(needed_zips, 1):
            obj_id = zip_to_id[inner_name]
            obj_dir = f'{img_dir}/{obj_id}'
            os.makedirs(obj_dir, exist_ok=True)
            
            inner_data = outer_z.read(inner_name)
            with zipfile.ZipFile(io.BytesIO(inner_data)) as inner_z:
                for img_file in inner_z.namelist():
                    if img_file.endswith(('.png', '.jpg')):
                        fname = os.path.basename(img_file)
                        with inner_z.open(img_file) as src, open(f'{obj_dir}/{fname}', 'wb') as dst:
                            dst.write(src.read())
            
            if i % 100 == 0 or i == len(needed_zips):
                print(f'  [{i}/{len(needed_zips)}]')
    
    # Xóa main zip để tiết kiệm disk
    if os.path.exists(main_zip_path):
        os.remove(main_zip_path)
        print('✓ Main zip deleted to save disk')
    
    n_final = len([d for d in os.listdir(img_dir) if os.path.isdir(f'{img_dir}/{d}')])
    print(f'✓ Rendered images: {n_final}/{n_total} folders')

# --- Tạo splits (LUÔN TẠO LẠI KHI DATA ĐẦY ĐỦ) ---
print('\nTạo splits từ data đầy đủ...')
os.chdir(WORK_DIR)
# Xóa splits cũ để tạo mới
splits_dir = f'{DATA_DIR}/splits'
if os.path.exists(splits_dir):
    shutil.rmtree(splits_dir)
!python src/data/prepare_data.py --data-dir "{DATA_DIR}" --skip-extract

for split in ['train', 'val', 'test']:
    sp = f'{DATA_DIR}/splits/{split}.txt'
    if os.path.exists(sp):
        with open(sp) as f:
            n = len([l for l in f if l.strip()])
        print(f'  {split}: {n} objects')

!df -h /content | tail -1
print('\n✓ Data sẵn sàng!')

## 5. Cấu hình tối ưu cho Colab T4

Config được tối ưu cho GPU T4 với batch size 32, 100 epochs.

In [ ]:
import yaml, torch

DATA_DIR = f'{WORK_DIR}/Cap3D_data'

has_gpu = torch.cuda.is_available()
if has_gpu:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {device_name} ({vram_gb:.0f} GB VRAM)')
else:
    print('CPU mode')

import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'System RAM: {ram_gb:.0f} GB')

# === CONFIG TỐI ƯU CHO A100 80GB + 167GB RAM ===
colab_config = {
    'data': {
        'data_dir': DATA_DIR,
        'num_points': 2048,
        'image_size': 224,
        'batch_size': 64,
        'num_workers': 4,
        'max_objects': 0,
    },
    'model': {
        'clip_model': 'ViT-B-32',
        'pretrained': 'laion2b_s34b_b79k',
        'freeze_clip': True,
        'fusion_type': 'concat',
        'fused_dim': 512,
        'coarse_points': 256,
        'fine_points': 2048,
    },
    'training': {
        'epochs': 100,
        'lr': 0.001,
        'weight_decay': 0.01,
        'warmup_epochs': 5,
        'patience': 25,
        'cooldown_seconds': 0,
    },
    'loss': {
        'lambda_cd': 1.0,
        'lambda_emd': 0.5,
        'lambda_color': 0.5,
        'use_emd': False,
    },
    'logging': {
        'log_dir': 'runs',
        'checkpoint_dir': 'checkpoints',
        'log_every': 10,
        'val_every': 1,
    }
}

config_path = f'{WORK_DIR}/configs/colab.yaml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(colab_config, f, default_flow_style=False)

n_train = 6343
batches = n_train // 64
print(f'\n✓ Config saved:')
print(f'  batch=64, points=2048, epochs=100')
print(f'  {batches} batches/epoch')

## 6. Training

**Chống Colab timeout:** Chạy cell Keep-Alive TRƯỚC khi train.  
Checkpoints tự động lưu mỗi 50 batches trên Drive.

In [ ]:
# === CHỐNG COLAB TIMEOUT ===
# Chạy cell này TRƯỚC khi train. Nó sẽ chạy ngầm và giữ session sống.
# Cách hoạt động: inject JavaScript vào browser để click tự động mỗi 60 giây

from IPython.display import display, Javascript
display(Javascript('''
function KeepAlive() {
    console.log("Keep-alive: " + new Date().toLocaleTimeString());
    // Click vào nút connect nếu bị disconnect
    var buttons = document.querySelectorAll("colab-connect-button");
    if (buttons.length > 0) { buttons[0].click(); }
    // Click vào toolbar để giữ session active
    var toolbar = document.querySelector("#toolbar");
    if (toolbar) {
        var event = new MouseEvent('click', { bubbles: true });
        toolbar.dispatchEvent(event);
    }
}
// Chạy mỗi 60 giây
setInterval(KeepAlive, 60000);
console.log("✓ Keep-alive started (60s interval)");
'''))
print('✓ Keep-alive đã bật! Session sẽ không bị timeout.')

In [ ]:
%%time
# === BẮT ĐẦU TRAINING ===
import os, torch
os.chdir(WORK_DIR)

# Kiểm tra checkpoint để resume (xử lý file hỏng + model không khớp)
CKPT_DIR = f'{WORK_DIR}/checkpoints'
resume_path = ''

for ckpt_name in ['last_model.pth', 'best_model.pth']:
    ckpt_file = f'{CKPT_DIR}/{ckpt_name}'
    if os.path.exists(ckpt_file):
        try:
            ckpt = torch.load(ckpt_file, map_location='cpu')
            # Kiểm tra config trong checkpoint có khớp với config hiện tại
            ckpt_cfg = ckpt.get('config', {})
            ckpt_points = ckpt_cfg.get('model', {}).get('fine_points', 0)
            import yaml
            with open(f'{WORK_DIR}/configs/colab.yaml') as f:
                cur_cfg = yaml.safe_load(f)
            cur_points = cur_cfg.get('model', {}).get('fine_points', 0)
            
            if ckpt_points == cur_points:
                resume_path = ckpt_file
                print(f'✓ Checkpoint tương thích: {ckpt_name}')
                break
            else:
                print(f'✗ Checkpoint không tương thích ({ckpt_name}): fine_points={ckpt_points} vs {cur_points}')
                print(f'  Xóa checkpoint cũ...')
                os.remove(ckpt_file)
        except Exception as e:
            print(f'✗ Checkpoint hỏng: {ckpt_name} → Xóa')
            os.remove(ckpt_file)

if resume_path:
    print(f'Resume từ: {resume_path}')
else:
    print('Train từ đầu (không có checkpoint tương thích)')

resume_flag = f'--resume "{resume_path}"' if resume_path else ''

!python src/train.py \
    --config configs/colab.yaml \
    {resume_flag}

In [ ]:
# Kiểm tra checkpoints đã lưu
import os
CKPT_DIR = f'{WORK_DIR}/checkpoints'
for f in ['best_model.pth', 'last_model.pth']:
    p = f'{CKPT_DIR}/{f}'
    if os.path.exists(p):
        size = os.path.getsize(p) / (1024**2)
        print(f'✓ {f} ({size:.1f} MB)')
print('\nTraining hoàn tất! Checkpoints đã trên Drive.')

## 7. Evaluation

Đánh giá model trên test set bằng Chamfer Distance và F-Score.

In [ ]:
os.chdir(WORK_DIR)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

!python src/evaluate.py \
    --config configs/colab.yaml \
    --checkpoint checkpoints/best_model.pth \
    --split test \
    --output results/evaluation.json

In [ ]:
# Xem kết quả
import json
with open(f'{WORK_DIR}/results/evaluation.json') as f:
    results = json.load(f)
print('=== KẾT QUẢ EVALUATION ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

## 8. Demo Visualization

Tạo 3D point cloud từ ảnh và mô tả văn bản.

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

import torch
import numpy as np
import yaml
from PIL import Image
from torchvision import transforms
import plotly.graph_objects as go

from src.models.multimodal3d import build_model

# Load config & model
with open(f'{WORK_DIR}/configs/colab.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(config).to(device)

ckpt = torch.load(f'{WORK_DIR}/checkpoints/best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✓ Model loaded (epoch {ckpt["epoch"]}, best_cd={ckpt["best_val_cd"]:.4f})')

In [ ]:
# === TEST VỚI ẢNH TỪ TEST SET ===
from google.colab import files

# Lấy object đầu tiên từ test set
test_split = f'{WORK_DIR}/Cap3D_data/splits/test.txt'
with open(test_split) as f:
    test_ids = [l.strip() for l in f if l.strip()]

obj_id = test_ids[0]
img_dir = f'{WORK_DIR}/Cap3D_data/rendered_images/{obj_id}'
img_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.png') and '_depth' not in f and '_MatAlpha' not in f])
img_path = os.path.join(img_dir, img_files[0])
print(f'Object: {obj_id}')
print(f'Image: {img_path}')

# Load và hiển thị ảnh
img = Image.open(img_path).convert('RGB')
display(img.resize((224, 224)))

# Tiền xử lý
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711]),
])
img_tensor = preprocess(img).unsqueeze(0).to(device)
text = 'a 3D object'
text_tokens = model.tokenize([text]).to(device)

# Generate 3D
with torch.no_grad():
    outputs = model(img_tensor, text_tokens)

pc = outputs['point_cloud'][0].cpu().numpy()
xyz = pc[:, :3]
rgb = np.clip(pc[:, 3:], 0, 1)

# Visualize
fig = go.Figure(data=[go.Scatter3d(
    x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
    mode='markers',
    marker=dict(
        size=3,
        color=[f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b in rgb],
        opacity=0.9,
    )
)])
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
        aspectmode='data', bgcolor='rgb(30,30,30)',
    ),
    paper_bgcolor='rgb(30,30,30)',
    margin=dict(l=0, r=0, b=0, t=0), height=500,
)
fig.show()

## 9. Download Checkpoint

Tải checkpoint đã train về máy tính.

## 10. Ghi chú & Hướng dẫn

### Nếu Colab bị ngắt giữa chừng:
1. Reconnect Colab
2. Chạy lại Cell 1 (Mount Drive)
3. Chạy lại Cell cài dependencies
4. Chạy Cell Training → nó sẽ tự **resume** từ checkpoint trên Drive

### So sánh config:
| | PC (default.yaml) | Colab (colab.yaml) |
|---|---|---|
| batch_size | 8 | **32** |
| epochs | 50 | **100** |
| lr | 0.0005 | **0.001** |
| lambda_color | 0.2 | **0.5** |
| cooldown | 30s | **0** |
| num_workers | 0 | **2** |
| ~Thời gian/epoch | ~30 phút | **~3-5 phút** |